In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

---
## 1. Đọc dữ liệu

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## 2. EDA dữ liệu

In [3]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
train.shape

(8693, 14)

In [5]:
train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [6]:
train.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [7]:
train["Transported"].value_counts()

Transported
True     4378
False    4315
Name: count, dtype: int64

Sau khi EDA dữ liệu ta nhận thấy Dataset có nhiều missing value và dữ liệu chưa phù hợp để huấn luyện. Nên ta sẽ có bước Preprocessing Data.

---
## 3. Preprocessing Data

In [8]:
X = train.drop("Transported", axis=1)
y = train["Transported"]
all_data = pd.concat([X, test], ignore_index=True)
#PassengerID
all_data[["GroupID", "MemberID"]] = (
    all_data["PassengerId"]
    .str.split("_", expand=True)
)
#Cabin
all_data[["Deck", "CabinNum", "Side"]] = (
    all_data["Cabin"]
    .str.split("/", expand=True)
)
all_data["CabinNum"] = pd.to_numeric(
    all_data["CabinNum"],
    errors="coerce"
)
#fillna
numeric_cols = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "CabinNum"
]
for col in numeric_cols:
    all_data[col] = all_data[col].fillna(
        all_data[col].median()
    )
categorical_cols = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "Deck",
    "Side"
]
for col in categorical_cols:
    all_data[col] = all_data[col].fillna(
        all_data[col].mode()[0]
    )

#Spend, GroupId, Age
spending = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

all_data["TotalSpend"] = all_data[spending].sum(axis=1)

all_data["NoSpending"] = (
    all_data["TotalSpend"] == 0
).astype(int)

all_data["GroupSize"] = (
    all_data.groupby("GroupID")["GroupID"]
            .transform("count")
)

all_data["IsChild"] = (
    all_data["Age"] < 18
).astype(int)

all_data["LuxurySpend"] = (
    all_data["Spa"] +
    all_data["VRDeck"]
)

all_data["BasicSpend"] = (
    all_data["RoomService"] +
    all_data["FoodCourt"] +
    all_data["ShoppingMall"]
)
all_data["CabinNum"] = (
    all_data["CabinNum"] // 100
).astype(int)

Đầu tiên ta gộp tập train và tập test lại để xử lý vì có cùng cấu trúc. Tiếp theo ta dựa trên train.head() mà xử lý, ta thấy ID được chia bởi dấu "_" nên ta tách ra. Sau đó tương tự với những features khác. Đối với total spend ta gộp những khoảng chi cho những hạng mục vào cùng 1 biến để dễ xử lý, tương tự với no spend. Sau khi đã tách các features, tiếp theo ta sẽ xử lý missing value của các numberic features và categorical.

In [9]:
all_data.drop(
    columns=[
        "PassengerId",
        "Cabin",
        "Name",
        "GroupID",
        "MemberID"
    ],
    inplace=True
)
all_data.isnull().sum()

HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Deck            0
CabinNum        0
Side            0
TotalSpend      0
NoSpending      0
GroupSize       0
IsChild         0
LuxurySpend     0
BasicSpend      0
dtype: int64

Nhìn vào train.describe ta thấy ví dụ cột Roomservice có rất nhiều người chi rất ít (25%=0, 50%=0) nhưng lại có max = 14327. Rõ ràng đây chính là outlier, vì thế em quyết định dùng median để fill vào numberic features vì median ít nhạy với outlier hơn mean. Còn đối với categorical thì sau khi em tách các thành phần thành features mới em quyết định drop các features đã không còn sử dụng sau đó fillna bằng mode. Cuối cùng kiểm tra lại ta thấy Data đã hết missing value và sẵn sàng đến bước tiếp theo.

## 4. Tách tập train và test để bắt đầu train

In [10]:
all_data = pd.get_dummies(
    all_data,
    drop_first=True
)
all_data.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,CabinNum,TotalSpend,NoSpending,GroupSize,...,Destination_TRAPPIST-1e,VIP_True,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_S
0,39.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1,1,...,True,False,True,False,False,False,False,False,False,False
1,24.0,109.0,9.0,25.0,549.0,44.0,0,736.0,0,1,...,True,False,False,False,False,False,True,False,False,True
2,58.0,43.0,3576.0,0.0,6715.0,49.0,0,10383.0,0,2,...,True,True,False,False,False,False,False,False,False,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,0,5176.0,0,2,...,True,False,False,False,False,False,False,False,False,True
4,16.0,303.0,70.0,151.0,565.0,2.0,0,1091.0,0,1,...,True,False,False,False,False,False,True,False,False,True


In [11]:
X = all_data.iloc[:len(train)]
X_test = all_data.iloc[len(train):]
print(X.shape)
print(X_test.shape)

(8693, 27)
(4277, 27)


In [12]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Decision Tree

In [13]:
dt = DecisionTreeClassifier(
    random_state=42
)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_valid)
print("Decision Tree Accuracy:",
      accuracy_score(y_valid, pred_dt))

Decision Tree Accuracy: 0.7475560667050029


# Bagging

In [14]:
rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=15,
    min_samples_leaf=2,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_valid)
print("Random Forest Accuracy:",
      accuracy_score(y_valid, pred_rf))

Random Forest Accuracy: 0.8067855089131685


# Gradient Boosting

In [15]:
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_valid)
print("Gradient Boosting Accuracy:",
      accuracy_score(y_valid, pred_gb))

Gradient Boosting Accuracy: 0.8073605520414031


# Stacking

In [16]:
estimators = [
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )),
    ("gb", GradientBoostingClassifier(
        random_state=42
    ))
]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000)
)
stack.fit(X_train, y_train)
pred_stack = stack.predict(X_valid)
print("Stacking Accuracy:",
      accuracy_score(y_valid, pred_stack))

Stacking Accuracy: 0.8073605520414031


---
Ta so sánh tất cả mô hình với nhau để chọn ra mô hình tốt nhất

In [17]:
result = {
    "Decision Tree": accuracy_score(y_valid, pred_dt),
    "Random Forest": accuracy_score(y_valid, pred_rf),
    "Gradient Boosting": accuracy_score(y_valid, pred_gb),
    "Stacking": accuracy_score(y_valid, pred_stack)
}
for model, score in result.items():
    print(f"{model}: {score:.4f}")

Decision Tree: 0.7476
Random Forest: 0.8068
Gradient Boosting: 0.8074
Stacking: 0.8074


In [18]:
stack.fit(X, y)
test_pred = stack.predict(X_test)

## 5. Tạo submission lên kaggle

In [19]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_pred
})
submission.head()
submission.to_csv(
    "submission.csv",
    index=False
)

---
## 6. Kết luận
Trong bài toán dự đoán Spaceship Titanic, các kỹ thuật Ensemble đều cho kết quả tốt hơn mô hình Decision Tree cơ sở. Điều này cho thấy việc kết hợp nhiều mô hình giúp giảm sai số và tăng khả năng tổng quát hóa trên dữ liệu chưa từng thấy.

Bên cạnh việc lựa chọn mô hình, tiền xử lý dữ liệu và Feature Engineering cũng có ảnh hưởng rất lớn đến kết quả. Việc xử lý giá trị thiếu, tách thông tin từ Cabin và PassengerId, xây dựng các đặc trưng như TotalSpend, GroupSize và NoSpending đã giúp mô hình khai thác tốt hơn các đặc điểm của dữ liệu.

Sau khi so sánh các mô hình, mô hình có độ chính xác cao nhất được lựa chọn để huấn luyện trên toàn bộ tập huấn luyện và tạo file submission.csv để dự đoán trên tập kiểm tra của cuộc thi Kaggle. Kết quả trên leaderboard đạt 0.80+, đáp ứng yêu cầu của bài tập và cho thấy hiệu quả của các kỹ thuật Ensemble trong bài toán phân loại này.